# Fit iterated MIMIC image embeddings

Load a serialized vision dataset, fit a stacked `IteratedMIMIC`, save the top-level embeddings, and save the fitted iterated model.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "mimic_vision").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import IteratedMIMIC
from mimic_vision import (
    VisionEmbedding,
    load_serialized_vision_dataset,
    resolve_artifact_file,
    save_vision_embedding,
)

In [ ]:
DATASET_FILE = "last"  # Use "last" or an explicit dataset filename.
VISION_DATA_DIR = PROJECT_ROOT / "data" / "vision"
DATASET_DIR = VISION_DATA_DIR / "serialized"
EMBEDDING_DIR = VISION_DATA_DIR / "embeddings"
MODEL_DIR = VISION_DATA_DIR / "models"

N_STEPS = 2
BASE_LEVEL = {"mode": "direct", "capacity": 0.2, "bootstrap": False, "feature_n_jobs": -1}
HIGHER_LEVEL = {"mode": "direct", "capacity": 0.1, "bootstrap": False, "feature_n_jobs": -1}
LEVELS = None  # Optional full list of per-level MIMIC constructor specs.
RANDOM_STATE = 0

In [ ]:
dataset_path = resolve_artifact_file(DATASET_FILE, input_dir=DATASET_DIR, pattern="*.pkl")
DATASET_FILE = dataset_path.name
dataset = load_serialized_vision_dataset(dataset_path)
DATASET_FILE, dataset.X.shape, dataset.images.shape, dataset.y.value_counts().sort_index()

In [ ]:
base_level = dict(BASE_LEVEL)
base_level["columns"] = {
    "ignore": [],
    "classification": [],
    "regression": list(dataset.X.columns),
}

model = IteratedMIMIC(
    n_steps=N_STEPS,
    base_level=base_level,
    higher_level=HIGHER_LEVEL,
    levels=LEVELS,
    random_state=RANDOM_STATE,
)
model.fit(dataset.X)

In [ ]:
top_embeddings = model.transform(dataset.X)
level_shapes = [tuple(frame.shape) for frame in model.representations_]
top_embeddings.shape, level_shapes

In [ ]:
embedding_artifact = VisionEmbedding(
    dataset_file=DATASET_FILE,
    embeddings=top_embeddings,
    mode=f"iterated{len(model.models_)}",
    capacity=float(HIGHER_LEVEL.get("capacity", 0.0)),
    random_state=RANDOM_STATE,
)
saved_embedding_path = save_vision_embedding(embedding_artifact, output_dir=EMBEDDING_DIR)
saved_embedding_path.name

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_filename = saved_embedding_path.with_suffix(".joblib").name
saved_model_path = MODEL_DIR / model_filename
model.save(saved_model_path)
saved_model_path.name